# Where CMA-ES and BO fail on a needle landscape, and what fixes it

**Research question.** At a fixed budget of N = 300 evaluations, how do population size, restarts and acquisition choices change the probability that CMA-ES and GP-BO find a narrow or distant global basin, and why?

**Convention.** This notebook only loads saved results (`results/summary*/`, `results/raw*/`) and draws them. Runs, aggregation and figure code live in `run_experiments.py` and `code/` (`analysis.py`, `report.py`). Findings live here, in markdown: prediction, result with CI, explanation, caveat.

- `FAST = True`: smoke test, N = 100, 5 seeds, from `results/*_fast/`. Every figure drawn from it carries a "SMOKE TEST" stamp.
- `FAST = False`: full results (N = 300, 50 seeds) from `results/summary/`.

The status table says which experiments have results and gives the command to produce the rest.

In [ ]:
FAST = True   # True: smoke test (N=100, 5 seeds) from results/*_fast.  False: full results from results/summary.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd().resolve()   # find the repo whatever folder the kernel starts in (notebooks/, repo root, or its parent)
ROOT = next(p for p in [here, *here.parents, *(d for d in here.iterdir() if d.is_dir())] if (p / "code" / "report.py").exists())
sys.path.insert(0, str(ROOT / "code"))
import report
M = report.Mode(FAST)
print(f"python = {sys.executable}  (should end in DSA4212/.venv/bin/python)")
print(f"mode = {'FAST (smoke test)' if FAST else 'FULL'}; N = {M.N}; seeds per cell = {M.n_seeds}; summaries: {M.summary_dir.relative_to(ROOT)}")

## 0. Status: which experiments have results

In [ ]:
report.status_table(M)

*Next: the headline findings, left open until the full runs exist.*

## 1. Summary

> **The FAST numbers in this notebook (N = 100, 5 seeds) are a smoke test of the pipeline, not results.** No finding below is written until the full run exists.

- **Baselines (Exp 0, full run, N = 300, 50 seeds):** default CMA-ES succeeded in 0/50 runs, default BO in 5/50 and random sampling in 2/50. The three Wilson intervals overlap pairwise, so the methods are not distinguishable from each other or from random sampling at this seed count. CMA-ES ended in the decoy in 48/50 runs; BO's 45 failures never sampled inside the well. See sections 3 and 4 (the explanations there are hypotheses).
- **Population size (Exp 1, full run, N = 300, 50 seeds):** success was 0/50 at $\lambda$ = 6, 12 and 24, so success does not change. Runs that put a sample inside the well rose from 1 to 7 to 13 of 50, all in the first few generations. See the Exp 1 card. Side check at N = 1500 (CMA-ES only, never on BO axes): still 0/50 at every $\lambda$, so the N = 300 result is not a budget cut-off.
- **Restarts (Exp 2, full run, N = 300, 50 seeds):** success was 0/50 for no restarts, IPOP and BIPOP. Restarts raised well entry from 1 to 6 of 50 runs but none reached $\varepsilon$. IPOP and BIPOP are identical runs at this N (one restart per run), so Exp 2 cannot separate them.
- **Initial step size (Exp 3, full run, N = 300, 50 seeds):** from the baseline random start no $\sigma_0$ in {0.3, 0.5, 1, 2, 4} gives more than 1/50. Started 1.0 from the well (oracle diagnostic), success falls from 33/50 at $\sigma_0 = 0.3$ to 2/50 at $\sigma_0 = 2$ (difference +0.62, interval excludes zero). Step size relative to the well width decides refinement; breadth decides finding. Follow-ups: the search mean walks into the well at $\sigma_0 = 0.3$ and away from it to the decoy at $\sigma_0 = 2$ (CMA4, supported side); success is *not* a function of $\sigma_0 / w$ alone, because wide wells tolerate a large $\sigma_0$ (CMA5, contradicted side).
- **BO exploration (Exp 4):** to fill in after the full run.
- **Unimodal control (Exp 6):** to fill in after Exps 1-5 choose the settings and Exp 6 runs.
- **Basin width (Exp 7):** to fill in after the full run (tests CMA3: CMA-ES success rises as the well widens toward σ₀).
- **Validation (Exp 8):** to fill in after Exps 1-5 choose the settings and Exp 8 runs.
- **Use X when Y:** to fill in after the full run (section 7).

*Next: the landscape and the definitions every later number rests on.*

## 2. Setup: the landscape, success, outcomes, seeds

**Primary landscape.** A broad decoy bowl centred at $c$ plus a narrow, deep Gaussian well at $x_g$:

$$f(x) = \frac{\lVert x - c\rVert^2}{s} \;-\; A\,\exp\!\left(-\frac{\lVert x - x_g\rVert^2}{2w^2}\right)$$

with $c = 0$, $s = 10$, $A = 4$, $w = 0.3$ and the far well at $x_g = (3, 3)$, on the domain $[-5, 5]^2$. Exp 7 varies $w$ and moves the well to $(2, 0)$ ("near").

**Success.** A run succeeds when its best value has regret $\le \varepsilon$, where regret is $f_{\text{best}} - f^*$. Because regret needs the known minimum $f^*$, it is available only on benchmark functions.

- **Primary:** $\varepsilon = 0.05$, chosen from random sampling only: the largest value on a fixed grid at which uniform random sampling at N = 300 succeeds in at most 5% of 200 pilot seeds (9000-9199). It gave 5/200 = 2.5%. The success region is a disc of radius about 0.05 around $x^*$ (`docs/figures/epsilon_check.png`).
- **Validation functions** use the same procedure with a cap below the nearest non-global minimum's regret, so success implies the global basin: Rastrigin $\varepsilon = 0.2$ (cap 0.995), Schwefel $\varepsilon = 5$ (cap 118.4; its random success sits exactly on the 5% boundary, 10/200).

**Outcome labels** (every run gets exactly one):

- **success**: regret $\le \varepsilon$.
- **stuck in decoy**: the best point found does not lead to the well (local descent from it ends in the decoy).
- **unfinished**: the best point is in the well's basin but not within $\varepsilon$ of the minimum, so the run found the basin and did not finish refining.

**Fixed settings (frozen before the big runs).** Budget N = 300 for every CMA-ES vs BO comparison, restarts and BO initial design included. CMA-ES: $\sigma_0 = 2$, start $x_0 \sim U[-4, 4]^2$, restart triggers tolx = 3e-3 and tolfun = 1e-5, restarts redraw $x_0$ in the full box. BO: Matern-5/2 ARD GP fitted by maximum likelihood with noise fixed at 1e-6, lengthscale bounds [0.05, 20] domain units, 10 initial space-filling points, refit every 5 iterations.

**Seeds, stated once.** Main experiments use seeds 1000-1049 (FAST uses the first five, 1000-1004). Fresh seeds 2000-2049 are used only for the Exp 7 best-variant line. Pilot seeds 9000-9199 are used only to choose $\varepsilon$ and the restart triggers. A seed fixes the CMA-ES start point and its random stream, and the BO initial design and Thompson draws, so every variant on seed $k$ starts from the same point or the same initial design. A run is reproducible from (method, config, landscape, seed).

**Visual language.** Same contour map throughout: **star** = true minimum, **cross** = decoy centre. CMA-ES is blue, BO is orange, random sampling is grey. Success rates are bars with Wilson 95% intervals and k/n above each bar.

In [ ]:
report.landscape_figure(M)

*Next: what happens when the two default optimisers meet this landscape.*

## 3. Failure: the Exp 0 baselines

First three **illustrations** of single runs (they show what a run looks like and prove nothing about rates), then the aggregate over seeds.

**CMA-ES, seed 1000.** Samples coloured by generation (light = early), the diamond is the start, red ellipses are the 1-sigma search distribution at a few generations, and the right panel is sigma per generation. The path is re-run from the same seed and checked against the saved CSV (a warning appears on the figure if they differ).

In [ ]:
report.cmaes_illustration(M)

**BO, seed 1000.** The 10 initial design points are white squares and later points are coloured by the order they were chosen.

In [ ]:
report.bo_illustration(M)

**Four seeds each (1000-1003).** The first four seeds of the fixed list, not picked for how they look. Top row CMA-ES, bottom row BO, colour = evaluation order.

In [ ]:
report.seed_grid(M)

**Aggregate over seeds.** Success rate with Wilson 95% intervals (left) and what happened to each run (right). Difference rows use a Newcombe interval, which is unpaired and so conservative when seeds are matched.

In [ ]:
report.show_predictions(["H1"])
report.exp0_figure(M)

*The numbers below are from the saved full run (N = 300, 50 seeds, seeds 1000-1049; set `FAST = False` to see its figures). The FAST figures above are a smoke test.*

**Prediction H1** (shown above): default CMA-ES gets stuck in a decoy basin.

**Result (with CI and effect size).** On `primary_far_w0.3`, $\varepsilon = 0.05$, N = 300:

| method | success | Wilson 95% | stuck in decoy | unfinished | median regret |
| --- | --- | --- | --- | --- | --- |
| CMA-ES default | 0/50 | [0.000, 0.071] | 48 | 2 | 2.208 |
| BO default (EI, $\xi$ = 0) | 5/50 | [0.043, 0.214] | 45 | 0 | 2.208 |
| random sampling | 2/50 | [0.011, 0.135] | 11 | 37 | 1.114 |

Differences in success rate (Newcombe 95%, unpaired so conservative): BO minus CMA-ES +0.10 [+0.01, +0.21]; BO minus random +0.06 [-0.05, +0.18]; CMA-ES minus random -0.04 [-0.13, +0.04].

By the rule "claim a difference only when the intervals separate", **the three methods are not distinguishable from one another at 50 seeds, random sampling included**: all three Wilson intervals overlap pairwise. The BO minus CMA-ES interval only just excludes zero and is a weaker statement; it does not license ranking them. What the data do show is that none of the three usually succeeds: even BO's upper limit is 21%, about one run in five.

**H1.** On each of H1's "supported by" criteria the measurements fall on the supported side: low success (0/50); best point outside the global basin in 48 of 50 CMA-ES runs; sigma collapse (section 4). Nothing on the "contradicted by" side was seen (success is not high and sigma does not stay large). The pre-registration does not define "early" collapse, so section 4 reports the numbers and the reading is left to you. The status in `docs/scoreboard.csv` is yours to set.

**Explanation** (hypotheses, not tested here):

- *Hypothesis:* the decoy bowl slopes toward its centre from everywhere, so a CMA-ES run that does not sample the well in its first generations has no signal pointing at it; the search distribution then contracts on the decoy. The two "unfinished" runs fit this picture: an early sample (generation 0 and generation 6) landed on the well's flank, yet sigma still collapsed onto the decoy.
- *Hypothesis:* BO and random sampling succeed mainly when a space-filling sample happens to land in the well. Of BO's five successes, three first entered the well within the 10-point initial design (evaluations 5, 5 and 9), one at evaluation 20 and one at evaluation 115. Measured, not tested: in the 45 failures no sample ever entered the well.

**Caveat.** One landscape (well width 0.3, far well) and one $\varepsilon$; 50 seeds leaves wide intervals, so small differences are unresolved. The settings are frozen ($\sigma_0 = 2$, start $U[-4,4]^2$, BO with 10 initial points, EI with $\xi = 0$, lengthscale bound 20) and results may depend on them. BO reproduces in distribution, not bit for bit, so the exact count 5/50 could shift on a re-run. "Unfinished" means the best point lies in the well's basin, not that the run was converging there (random sampling's 37 are best points on the well's flank).

*Next: the diagnostics that say why the baselines behave this way.*

## 4. Why: diagnostics

Two measured quantities, one per method. The mechanisms behind them are **hypotheses** until tested.

**CMA-ES step size.** Sigma per generation for every seed. *Hypothesis (from `docs/direction.md` section 2, not tested here):* inside a basin consecutive steps are anticorrelated, cumulative step-size adaptation shrinks sigma, and the search contracts and cannot leave. Measured: sigma, and where the best point ends up (section 3 outcomes). H1 is supported only by low success together with early sigma collapse and a best point outside the global basin.

In [ ]:
report.sigma_figure(M)

**BO fitted lengthscale versus the well width $w$.** Left: the smallest fitted lengthscale along a run on the primary landscape, against $w$ and the fitting upper bound. Right: its final value against $w$ across the far-well widths. *Hypothesis (not tested here):* when the lengthscale is much longer than $w$ the GP smooths over the well, gives it low uncertainty, and EI never samples it. Measured: the lengthscale and whether samples stay spread out; a causal claim would need a test that changes the lengthscale and nothing else. A lengthscale pinned at the upper bound also reflects the bound chosen before the runs.

In [ ]:
report.lengthscale_figure(M)

*The numbers below are from the saved full run (N = 300, 50 seeds); set `FAST = False` to see its figures.*

**Result (measured).**

- *CMA-ES step size.* Sigma starts at 2, rises above it in the first generations (median peak 2.24), then decays steadily in every run. It first falls below 0.2 (a tenth of the start value) at generation 19 (median; range 14-31) of 50, and below 0.02 at generation 35 (range 29-49). Median final sigma is 0.0025. From evaluation 100 on, at least 97.5% of every run's samples lie within 0.5 of the decoy centre, and the closest any CMA-ES sample comes to the well centre is 1.4 (median over runs).
- *BO lengthscale.* In 42 of 50 runs the smallest fitted lengthscale reaches the upper bound (20, twice the domain width), first at evaluation 15 (median; range 15-130), and it ends there. None of those 42 succeeded. Five of the other eight runs succeeded: their final lengthscale is 0.05-0.98, i.e. 0.17 to 3.3 times the well width $w = 0.3$.
- *BO sampling.* In the 45 failures no sample ever entered the well (closest approach 1.41, median), and about 99% of evaluations from 100 on are within 0.5 of the decoy centre. All 5 runs that did put a sample inside the well reached $\varepsilon$.

**Explanation** (hypotheses, not tested here):

- *CMA-ES hypothesis (from `docs/direction.md` section 2):* cumulative step-size adaptation shrinks sigma when consecutive steps are anticorrelated, as they are inside a basin, so the search contracts and cannot leave. The steady geometric decay in all 50 runs is consistent with this; no run held sigma fixed, so the mechanism itself is untested. Whether collapse is "early" is a reading of the numbers above (in the median run sigma is below 0.2 after 38% of the generations and below 0.02 after 70%), not a pre-registered threshold.
- *BO hypothesis A:* a lengthscale much longer than $w$ makes the surrogate smooth over the well, so EI never samples it.
- *BO hypothesis B (the reverse direction):* a run that has not yet sampled the well only ever sees the smooth bowl, so maximum likelihood picks a long lengthscale; the long lengthscale is then a symptom of missing the well, not its cause. The data cannot separate A from B: runs with a long lengthscale are exactly the runs without a well sample.
- *What would test them:* changing the lengthscale (for example capping it) while holding everything else fixed, or raising exploration ($\xi$, $\beta$) and checking whether samples reach the well; the latter is Exp 4.

**Caveat.** Where the lengthscale sits at the bound, the likelihood wanted a longer value than the frozen bound allows, so "at the bound" is partly a property of the bound. The BO split (all 5 runs with a well sample succeeded, none of the 45 others sampled it) says discovery, not refinement, is where BO fails here, but with 5 successes it is a description of this baseline, not an estimate of a rate. BO is reproducible in distribution only.

*Next: one factor at a time, to see which design choices change these failures.*

## 5. Experiment cards

One factor changes per experiment; every variant uses the same N and the same seeds. Each card has the same five parts: the pre-registered **prediction** (read from `docs/predictions.md`, unedited), a **figure** (success rate with Wilson 95% interval, same style as section 3, plus a table with the difference against the baseline), the **result** with CI and effect size, the **explanation**, and the **caveat**. A card's result and explanation are filled in only when its full results exist.

## CMA-ES block: Exps 1 to 3

Three CMA-ES design choices, one factor at a time: population size, restarts, then the initial step size that Exps 1 and 2 point to.

### Exp 1: population size $\lambda$ (1x, 2x, 4x default: 6, 12, 24)

Produce results: `python run_experiments.py --full --exp exp1 && python run_experiments.py --aggregate`

Also report generations completed per $\lambda$ (the table shows the median).

In [ ]:
report.show_predictions(['H2', 'M1'])
report.experiment_card("exp1", M)

*Full run: N = 300, 50 seeds (1000-1049), `primary_far_w0.3`, $\varepsilon = 0.05$. Set `FAST = False` to see the figure.*

**Result (with CI and effect size).**

| setting | success | Wilson 95% | samples ever inside the well ($d < w$) | best point in the well's basin | median generations |
| --- | --- | --- | --- | --- | --- |
| default ($\lambda$ = 6) | 0/50 | [0.000, 0.071] | 1 | 2 | 50 |
| $\lambda$ = 12 | 0/50 | [0.000, 0.071] | 7 | 7 | 25 |
| $\lambda$ = 24 | 0/50 | [0.000, 0.071] | 13 | 16 | 13 |

Success is 0/50 at every $\lambda$, so the differences in success are exactly zero and the intervals coincide. What does change with $\lambda$ is how often the search touches the well: 1, 7 and 13 of 50 runs put at least one sample inside it, and the median closest approach to the well centre falls from 1.40 to 1.03 to 0.65. Every first well sample came in generation 6 or earlier (generation 4 or earlier at $\lambda = 24$), and a run that entered the well had one to three samples in it in total. No run that entered went on to reach $\varepsilon$.

**H2** (larger $\lambda$ raises success, with a sweet spot): on success itself the data fall on the "contradicted by" side (no change). The statuses in `docs/scoreboard.csv` are yours to set. **M1** (any $\lambda$ gain comes only from extra early samples): the one quantity that rises with $\lambda$, well entry, happens in the first few generations, which is the pattern M1 describes. Neither status is set here.

**Explanation** (hypotheses, not tested here):

- *Hypothesis:* a larger population draws more samples before the search distribution contracts (24 in generation 0 against 6), so it is likelier to drop one in a well that is 0.3 wide in a box 10 wide. This is extra coverage, not selection averaging, because the bowl slopes toward the decoy and there is no global trend toward the well to follow.
- *Hypothesis:* touching the well is not enough. With $\sigma_0 = 2$ against $w = 0.3$, one or two good samples among 6 to 24 weigh little in the update against the many samples near the decoy, so the mean keeps moving to the decoy and the well sample is never refined. The engine itself is not at fault in general (the sphere sanity check solved 20/20), but this particular mechanism is untested; Exp 2 (restarts) and the $\sigma$ traces are where to look.

**Caveat.** A floor of 0/50 cannot rank the settings on success; the well-entry counts are descriptive (1, 7, 13 are small counts) and were not pre-registered as a metric. One landscape, one $\varepsilon$, frozen $\sigma_0 = 2$ and start region. At $\lambda = 24$ a run completes only 13 generations, so any longer-run benefit would be cut off by the fixed budget.

#### Exp 1 side check: is $\lambda$ = 24 simply cut off by the budget?

**Why this was added.** Exp 1 gave 0/50 at every $\lambda$ at N = 300. At the end of those runs $\lambda$ = 6 and 12 had already contracted (median final $\sigma$ 0.0025 and 0.076), so more evaluations could not help them. $\lambda$ = 24 had not (median final $\sigma$ = 1.13 after only 13 generations), so its 0/50 could mean either "cannot succeed" or "ran out of budget before it could". To tell these apart we repeat the same three settings, on the same seeds, with five times the budget (N = 1500). This was decided before it ran (`docs/pilot_log.md`), with one extra budget only.

**How it is kept honest.** It is a side check, not part of any CMA-ES vs BO comparison: N = 1500 results live in `results/summary_long/`, are never drawn on BO axes, and no BO run is compared to them. The main comparisons use N = 300 throughout, because that is the budget every method is held to.

Run it with `python run_experiments.py --long --exp exp1_long && python run_experiments.py --aggregate --long`.

In [ ]:
report.long_budget_card(M)

**Result.** Success is 0/50 at every $\lambda$ even with five times the budget. The number of runs whose best point lies in the well's basin is also unchanged (2, 7 and 16 of 50): the extra 1200 evaluations found the well in no new run. By the end $\lambda$ = 24 has contracted as well (median final $\sigma$ 0.0034, after a median 63 generations).

**What it shows.** The 0/50 at N = 300 is not a budget effect. The searches that did not touch the well early did not find it later, and the ones that touched it did not refine it. This matches the picture from Exp 0 and Exp 1 that CMA-ES here is decided in the first few generations (extra early samples, hypothesis M1), not by running longer. It also means a bigger $\lambda$ cannot be rescued by a bigger budget on this landscape.

**Caveat.** One extra budget (5x), one landscape, one $\varepsilon$. A much longer budget was not tried, and by design it would not be comparable to BO at N = 300 anyway.

### Exp 2: restarts (none, IPOP, BIPOP; total budget fixed at N)

Produce results: `python run_experiments.py --full --exp exp2 && python run_experiments.py --aggregate`

Restart triggers are frozen at tolx = 3e-3, tolfun = 1e-5. **Must report** restart firing rate per $\lambda$ (pilot, no-restart runs: 100% at $\lambda$ = 6, 84% at 12, 0% at 24). At $\lambda$ = 24 restarts never fire, so IPOP/BIPOP started there equal no restarts.

In [ ]:
report.show_predictions(['H3', 'M2'])
report.experiment_card("exp2", M)

*Full run: N = 300, 50 seeds (1000-1049), `primary_far_w0.3`, $\varepsilon = 0.05$. Set `FAST = False` to see the figure.*

**Result (with CI and effect size).**

| setting | success | Wilson 95% | runs with a sample in the well ($d < w$) | best point in well basin | restarts per run |
| --- | --- | --- | --- | --- | --- |
| no restarts (default) | 0/50 | [0.000, 0.071] | 1 | 2 | 0 |
| IPOP | 0/50 | [0.000, 0.071] | 6 | 9 | 1 (every run) |
| BIPOP | 0/50 | [0.000, 0.071] | 6 | 9 | 1 (every run) |

Success is 0/50 for all three, so the differences are exactly zero. Restarts did raise how often the search touched the well, from 1 to 6 of 50 runs, but none reached $\varepsilon$. Five of the six well entries came from the restart ($\lambda$ = 12, started at a fresh $x_0 \sim U[-5,5]^2$); the sixth is the seed whose first start had already entered in Exp 0 and Exp 1.

**IPOP and BIPOP are the same here, by construction, not by accident.** All 50 BIPOP runs are identical to the IPOP runs, evaluation by evaluation. BIPOP's first restart is always the large-population regime, which is IPOP's first doubling, and both use the same restart random stream. At N = 300 each run has only one restart: the first (default) start spends a median 186 of the 300 evaluations before its stopping rule fires (range 144-234), so the restart gets the remaining 114, enough for about nine generations at $\lambda$ = 12 and never enough to converge and trigger a second one. The two variants could only differ if a second restart were possible. This is a property of the fixed budget, not a bug (the restart bookkeeping is in `code/optimizers.py`), and it means Exp 2 cannot say anything about IPOP versus BIPOP at this N.

**H3** (restarts beat a larger $\lambda$ at equal budget): restarts reached 0/50, as did $\lambda$ = 12 and $\lambda$ = 24 in Exp 1, so on success the data fall on the "contradicted by" side (equal success). **M2** (a restart acts like random search: success close to what independent fresh starts predict): the first clause is the one this run can speak to. A fresh $\lambda$ = 12 start touched the well in 7 of 50 runs in Exp 1 (300 evaluations) and the restart touched it in 5 of 50 (114 evaluations), which is the same order, so restarts look like fresh starts in how they find the well. The second clause (which restart succeeded) cannot be assessed, because nothing succeeded. Neither status is set here; `docs/scoreboard.csv` is yours.

**Explanation** (hypotheses, not tested here):

- *Hypothesis:* restarts help find the well only as much as extra independent draws do, which is the M1 story again (coverage), and once the well is touched the same problem as in Exp 1 applies: $\sigma_0 = 2$ is large against $w = 0.3$, so the touch is not refined.
- *Hypothesis:* a restart that is expensive in evaluations (the default start takes about 186 of them) buys very little breadth at N = 300, so any restart advantage would need a smaller budget share for the first start.

**Caveat.** One landscape, one $\varepsilon$, frozen restart triggers (tolx = 3e-3, tolfun = 1e-5) and frozen $\sigma_0$. With a floor of 0/50 success cannot rank the settings; the well-entry counts (1, 6, 6) are small and were not pre-registered as a metric. The restart triggers were chosen from firing rates only, so they were not tuned to this outcome.

### What Exps 1 and 2 tell us, and why the initial step size and the well width matter

Exps 0, 1 and 2 (and the N = 1500 side check) all gave 0/50 for CMA-ES. Three measured facts, and one explanation to test.

**Measured.**

- More samples early help *find* the well: runs with a sample inside it went from 1/50 (λ = 6) to 7/50 (λ = 12) to 13/50 (λ = 24), always in the first few generations, and restarts raised it from 1/50 to 6/50.
- None of those runs reached ε. Finding the well did not turn into success, and extra budget did not either (N = 1500: still 0/50 and no new well found).
- The success criterion is reachable and the code works: in a positive control on *pilot* seeds (9000-9019, `pilot/cma_positive_control.py`, run after Exps 0-2 so it is post hoc), CMA-ES started at the well centre succeeded 20/20 with σ₀ = 0.3 and 17/20 with σ₀ = 0.5 from 1.0 away, but only 0/20 with the frozen σ₀ = 2, even when started exactly at the well.

**Explanation (hypothesis, tested next).** Success needs a point within about 0.05 of the well centre, and the well has width w = 0.3. With σ₀ = 2, about 7 times w, the early samples are spread so widely that the decoy bowl, which slopes toward its centre from everywhere, supplies most of the selected points. The update then moves the mean toward the decoy and σ contracts there, even if one sample landed in the well. In this picture λ and restarts cannot fix the failure because they change how often the well is *touched*, not whether it is *refined*, and the quantity that sets refinement is the ratio σ₀ / w.

**What is tested, in order.**

1. **Exp 3** changes σ₀ and nothing else, in two arms. A *random-start* arm (the baseline start) asks whether any σ₀ helps in practice (CMA2). An *oracle-start* arm, started 1.0 from the well, removes the finding problem to ask whether σ₀ alone sets refinement (CMA1). The oracle arm uses knowledge of the well, so it is a diagnostic of the mechanism and is never compared with BO.
2. **Exp 7** changes the well width w with σ₀ fixed. If the ratio σ₀ / w is what matters, CMA-ES success should rise as w grows toward σ₀ (CMA3).

If CMA1 holds the explanation is supported for the refinement step; if CMA2 also holds, no single σ₀ solves the problem from a random start, because a small σ₀ rarely reaches the well. Either way the current findings stand as measured.

### Exp 3: CMA-ES initial step size σ₀ (0.3, 0.5, 1, 2 default, 4)

Produce results: `python run_experiments.py --full --exp exp3 && python run_experiments.py --full --exp exp3_oracle && python run_experiments.py --aggregate`

Only σ₀ changes; λ is the default, there are no restarts, N = 300, seeds 1000-1049, `primary_far_w0.3`, ε = 0.05. **Random start** (the baseline U[-4,4]²) is the fair comparison. **Oracle start** (1.0 from the well centre, random direction) uses knowledge of the well, is a diagnostic only, and is not compared with BO. Prediction text is in `docs/predictions_addendum.md`.

In [ ]:
report.show_predictions(['CMA2'])
report.experiment_card("exp3", M)

*Full run: N = 300, 50 seeds (1000-1049), `primary_far_w0.3`, $\varepsilon = 0.05$, baseline random start $U[-4,4]^2$. Set `FAST = False` to see the figure.*

**Result (with CI and effect size).**

| $\sigma_0$ | success | Wilson 95% | runs with a sample in the well ($d < w$) | median closest approach to the well centre |
| --- | --- | --- | --- | --- |
| 0.3 | 0/50 | [0.000, 0.071] | 0 | 2.26 |
| 0.5 | 0/50 | [0.000, 0.071] | 0 | 2.09 |
| 1 | 1/50 | [0.004, 0.105] | 2 | 2.16 |
| 2 (default) | 0/50 | [0.000, 0.071] | 1 | 1.40 |
| 4 | 0/50 | [0.000, 0.071] | 4 | 0.93 |

No $\sigma_0$ changes success: 0 or 1 of 50 everywhere, every difference from the default is within about $\pm 0.1$ and every interval overlaps. What does change is how close the search gets to the well. A small $\sigma_0$ never touches it (0/50 runs at 0.3 and 0.5, closest approach about 2), because a tight cloud started at random settles on the nearest bowl; a large $\sigma_0$ touches it more often (4/50 at $\sigma_0 = 4$) and gets within 0.93 of its centre on median.

**CMA2** (no $\sigma_0$ gives high success from a random start; every arm stays below 12/50): the largest count is 1/50, so the measurements fall on the supported side. The status in `docs/scoreboard.csv` is yours.

**Explanation** (hypotheses, not tested here):

- *Hypothesis:* from a random start the two requirements pull $\sigma_0$ in opposite directions. Reaching the well needs breadth (large $\sigma_0$, more samples), and refining it needs a $\sigma_0$ comparable to $w$. The data show each half separately (well entry rises with $\sigma_0$; the oracle arm below shows refinement falls with it) but no value does both.
- *Hypothesis:* a small $\sigma_0$ makes CMA-ES a local search from its random start, so it succeeds only if the start is already near the well, which essentially never happens (the well covers about 0.3% of the box).

**Caveat.** One landscape, one $\varepsilon$, one start distribution, one grid of five values; a value between 0.3 and 1 combined with a larger $\lambda$ was not tried (that is a different factor combination). With counts of 0 and 1 the arms cannot be ranked on success.

**Oracle-start arm** (mechanism test; started 1.0 from the well centre).

In [ ]:
report.show_predictions(['CMA1'])
report.experiment_card("exp3_oracle", M)

*Full run: N = 300, 50 seeds (1000-1049), `primary_far_w0.3`, $\varepsilon = 0.05$. The first start is 1.0 from the well centre in a random direction (oracle start: it uses knowledge of the well, so this is a diagnostic of refinement, not a method comparison, and no BO run is compared with it). Set `FAST = False` to see the figure.*

**Result (with CI and effect size).**

| $\sigma_0$ | success | Wilson 95% | difference vs $\sigma_0 = 2$ (Newcombe 95%) | runs with a sample in the well |
| --- | --- | --- | --- | --- |
| 0.3 | 33/50 | [0.522, 0.776] | +0.62 [+0.45, +0.74] | 35 |
| 0.5 | 31/50 | [0.482, 0.741] | +0.58 [+0.41, +0.70] | 39 |
| 1 | 10/50 | [0.112, 0.330] | +0.16 [+0.03, +0.29] | 23 |
| 2 | 2/50 | [0.011, 0.135] | (baseline) | 11 |
| 4 | 1/50 | [0.004, 0.105] | -0.02 [-0.12, +0.07] | 11 |

Success falls steadily as $\sigma_0$ grows, 33, 31, 10, 2, 1 of 50, and the intervals at $\sigma_0 = 0.3$ and $\sigma_0 = 2$ do not overlap (difference +0.62, interval excluding zero by a wide margin). Starting from 1.0 away, the same CMA-ES that scored 0/50 from a random start succeeds in two thirds of runs when $\sigma_0$ is 0.3.

**CMA1** (started near the well, success falls as $\sigma_0$ grows; the $\sigma_0 = 0.3$ interval above the $\sigma_0 = 2$ interval): the measurements fall on the supported side. This is the same direction as the pilot positive control (pilot seeds, post hoc), now on the experiment seeds. The status in `docs/scoreboard.csv` is yours.

**Explanation** (hypotheses, not tested here):

- *Hypothesis:* with $\sigma_0 = 0.3$ (equal to $w$) the first samples are already inside or next to the well, the ranking points into it, and $\sigma$ contracts on it. With $\sigma_0 \ge 2$ most samples are decoy points that outrank nothing in the well, the bowl dominates the update, and the mean drifts to the decoy. The step size relative to $w$, not the distance to the well, decides which happens.
- Not every near-well run succeeds even at $\sigma_0 = 0.3$ (33 of 50, 35 of 50 touch the well): starting 1.0 away with a tight cloud, some runs do not reach it before contracting. That is the mirror of the random-start result: too small a $\sigma_0$ fails to find, too large fails to refine.

**Caveat.** This isolates refinement by giving CMA-ES an unfair start, so it shows what $\sigma_0$ does once the well is near, not what a user could achieve. One distance (1.0) and one landscape; the mechanism is shown by the dependence on $\sigma_0$, not proved: a direct check would track $\sigma$ and the mean per generation, which was not done. Exp 7 (width sweep) tests the same ratio from the other side.

#### Exp 3 follow-up A: where does the search mean go? (CMA4)

No new runs: the five oracle configs are re-run on seeds 1000-1049 (CMA-ES is deterministic given the seed) to read off, each generation, the distance of the search distribution's mean to the well centre. Command: `python run_experiments.py --trace`. This turns "success falls as $\sigma_0$ grows" into a direct look at the mechanism.

In [ ]:
report.show_predictions(['CMA4'])
report.mean_trace_figure(M)

*Full results (50 seeds); set `FAST = False` to see the figure.*

**Result.** With $\sigma_0 = 0.3$ the mean walks into the well: its median distance falls from 1.0 at generation 0 to 0.29 at generation 5, 0.09 at generation 10 and 0.01 at generation 20. With $\sigma_0 = 2$ it walks away: the median distance rises to 4.12 by generation 5 and sits at 4.2 from then on, which is the distance from the well $(3,3)$ to the decoy centre at the origin (4.24).

| setting | final mean within $w$ of the well | runs that touched the well | touched runs whose final mean is closer to the decoy centre than to the well |
| --- | --- | --- | --- |
| oracle $\sigma_0 = 0.3$ | 33/50 | 35 | 2/35 |
| oracle $\sigma_0 = 2$ | 1/50 | 11 | 10/11 |

**CMA4** (at $\sigma_0 = 0.3$ at least 60% of runs end within $w$; at $\sigma_0 = 2$ at most 15%, and in the touched $\sigma_0 = 2$ runs the mean ends nearer the decoy in more than half): all three conditions are met (66%, 2%, 10 of 11), so the measurements fall on the supported side. The status in `docs/scoreboard.csv` is yours.

**What it shows.** In 10 of the 11 runs at $\sigma_0 = 2$ where a sample did land in the well, the mean still ended at the decoy. A well sample is not enough: with a wide cloud the other selected points sit on the bowl and pull the mean away, which is the mechanism proposed above, now observed and not only inferred from success rates.

**Caveat.** The mean is the centre of the search distribution, not the best point; the trace is for the oracle start, one distance (1.0), one landscape. The thresholds in CMA4 were set after seeing Exp 3's success counts (not the traces).

#### Exp 3 follow-up B: does only the ratio $\sigma_0 / w$ matter? (CMA5)

The explanation says refinement is set by the step size *relative to the well*. If so, the same ratio should give the same success at every well width. Oracle start at $(10/3)\,w$ from the well (1.0 at $w = 0.3$, as in the oracle arm), $\sigma_0 = k\,w$ with $k \in \{1, 3, 7, 20\}$, on $w \in \{0.1, 0.3, 0.6, 1.0\}$, 50 seeds, N = 300. Command: `python run_experiments.py --full --exp exp3_ratio && python run_experiments.py --aggregate`. A diagnostic: it uses the oracle start and is never compared with BO.

In [ ]:
report.show_predictions(['CMA5'])
report.experiment_card("exp3_ratio", M)

*Full run: 50 seeds per cell (seeds 1000-1049), N = 300; set `FAST = False` to see the figure.*

**Result (Wilson 95% intervals above).** Success by $k = \sigma_0 / w$ and well width:

| $k$ | $w = 0.1$ | $w = 0.3$ | $w = 0.6$ | $w = 1.0$ |
| --- | --- | --- | --- | --- |
| 1 | 44/50 | 33/50 | 17/50 | 24/50 |
| 3 | 25/50 | 12/50 | 14/50 | 36/50 |
| 7 | 1/50 | 0/50 | 14/50 | 40/50 |
| 20 | 0/50 | 1/50 | 14/50 | 40/50 |

For the narrow wells ($w = 0.1, 0.3$) the pattern is Exp 3's: success collapses as $k$ grows (44 to 0, and 33 to 1). For the wide wells it does not. At $w = 0.6$ success is flat (17, 14, 14, 14) and at $w = 1.0$ it *rises* with $k$ (24 to 40).

**CMA5** had two conditions: success at $k = 1$ above $k = 20$ with non-overlapping intervals at every width, and the four widths not separated at $k = 1$. Both fail. At $w = 0.6$ the $k = 1$ and $k = 20$ intervals overlap ([0.22, 0.48] and [0.17, 0.42]); at $w = 1.0$ the order is reversed (24/50 against 40/50); and at $k = 1$ the intervals at $w = 0.1$ ([0.76, 0.94]) and $w = 0.6$ ([0.22, 0.48]) do not overlap. The measurements fall on the **contradicted** side. The status is yours to set.

**Check for a bug or slip (done before reading the result).** Several cells have identical counts (14, 14, 14 at $w = 0.6$; 40 and 40 at $w = 1.0$). The sample paths of $k = 7$ and $k = 20$ are in fact identical in all 50 seeds at $w = 0.6$ and at $w = 1.0$. This is a property of the `cma` library, not of our code: with bounds $[-5, 5]$ it caps the effective standard deviation (option `maxstd`, set from the bound range, about 4 here). Starting from the same point and seed, $\sigma_0 = 4, 5, 6, 7, 10, 20$ all draw exactly the same first samples, while $\sigma_0 = 3$ differs. So the labelled $k$ is not the effective ratio once $\sigma_0 > 4$: at $w = 0.6$ both $k = 7$ ($\sigma_0 = 4.2$) and $k = 20$ ($\sigma_0 = 12$) are effectively $\sigma_0 = 4$ (ratio 6.7); at $w = 1.0$ both are effectively 4 (ratio 4); the $k = 20$ arm at $w = 0.3$ ($\sigma_0 = 6$) is also effectively 4. Earlier experiments are not affected: Exps 1 and 2 use $\sigma_0 = 2$, Exp 3 reaches 4 at most (exactly at the cap), and Exp 3's oracle arm at $w = 0.3$ is unchanged. The logged $\sigma$ column shows the nominal value.

**What it shows (hypotheses, not tested).**

- Success is not a function of $\sigma_0 / w$ alone. For narrow wells a large $\sigma_0$ is fatal, as before; for wide wells it is not.
- *Hypothesis:* a wide well (at $w = 1.0$ the region below the decoy floor is about a tenth of the box) is found by a broad cloud and, being wide, has a funnel the cloud can follow, so refinement does not need a small $\sigma_0$. A small $\sigma_0$ at $w = 1.0$ starts 3.3 away and rarely reaches the well (finding fails), which is why $k = 1$ is the worst there. The decoy bowl has a fixed scale, so the well's size relative to the box and the bowl matters as well as the ratio.
- So the explanation needs refining: step size relative to the well matters when the well is small compared with the bowl and the box, and the width sweep (Exp 7) will show the same mixture from the other side.

**Caveat.** The effective-$\sigma$ cap means the high-$k$ arms at wide $w$ are not the ratios they are labelled with, and $k = 7$ and $k = 20$ are duplicates there; a cleaner ratio grid would stay at or below the cap ($\sigma_0 \le 4$) at every width. One oracle distance, one landscape, the same $\varepsilon$ at every width (it corresponds to roughly the same fraction of $w$, but the bowl does not scale).

## BO block: Exps 4 and 5

The two BO design choices, one factor at a time. Exp 5 follows Exp 4 directly.

### Exp 4: BO exploration vs exploitation (EI with $\xi$, LCB with $\beta$, Thompson sampling, MAP-fit arm)

Produce results: `python run_experiments.py --full --exp exp4 && python run_experiments.py --aggregate`

EI is swept over $\xi \in \{0, 0.01, 0.1, 0.5, 1, 2\}$ (units of $f$; the well is about 2.2 deep, so large $\xi$ is close to pure exploration) and LCB over $\beta \in \{0.5, 2, 8, 32\}$. The initial design is fixed at 10 points for every arm. `bo_map` changes the lengthscale prior and the noise treatment together, so it is a different fitting recipe, not a one-factor change.

Also report sample spread, the fitted lengthscale against $w$ (section 4) and the Cholesky jitter-fallback count.

**Link to Exp 0.** BO1 (post hoc, from Exp 0) says default BO fails at discovery; H4 tests whether more exploration fixes that; BO2 says any gain comes from discovery and not from a shorter lengthscale; BO3 uses the `bo_map` arm as a weak test of whether the long lengthscale is a cause or a symptom. Same shape as CMA-ES: H1, M1, H2. Prediction text is in `docs/predictions_addendum.md`.

In [ ]:
report.show_predictions(['H4', 'BO1', 'BO2', 'BO3'])
report.experiment_card("exp4", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp 5: BO kernel type (Matern-5/2 baseline, Matern-3/2, Matern-1/2, RBF)

Produce results: `python run_experiments.py --full --exp exp5 && python run_experiments.py --aggregate`

Only the kernel changes; EI with $\xi = 0$, 10 initial points and the lengthscale bounds on the unit cube are the baseline's. Report success given a well sample separately from overall success, and note that lengthscale values are not strictly comparable across kernels.

In [ ]:
report.show_predictions(['BO4'])
report.experiment_card("exp5", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

## Both methods: Exps 6, 7 and 8

Control, landscape knobs and validation. Best and worst variants are selected and evaluated separately for CMA-ES and for BO.

### Exp 6: unimodal control (sphere)

Produce results: `to be added to `code/configs.py` once Exps 1-5 choose the best and worst settings`

Figure when configured: median regret curves with quartiles, plus final regret at equal N per setting.

In [ ]:
report.show_predictions(['H5'])
report.experiment_card("exp6", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp 7: basin width

Produce results: `python run_experiments.py --full --exp exp7_baseline && python run_experiments.py --aggregate`

The far well runs at $w \in \{0.1, 0.3, 0.6, 1.0\}$ and the near well only at $w \le 0.3$ (the near well's basin merges with the decoy at $w \ge 0.6$). One fixed $\varepsilon$ is used throughout. The best-variant line will be selected by the rule in `docs/pilot_log.md` (written before Exps 1-5) and evaluated on fresh seeds 2000-2049; the baseline-only curves use seeds 1000-1049. **No prediction is pre-registered for Exp 7.**

In [ ]:
report.show_predictions(['CMA3'])
report.experiment_card("exp7_baseline", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp 8: validation on Rastrigin (funnel) and Schwefel (deceptive)

Produce results: `to be added to `code/configs.py` once Exps 1-5 choose the best and worst settings`

Figures when configured: success rate per function (separate panels) and the $\lambda$ gain (success-rate difference with a Newcombe interval, unpaired so conservative).

In [ ]:
report.show_predictions(['H6', 'M3', 'BO5'])
report.experiment_card("exp8", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

*Next: every pre-registered prediction in one table, with its status.*

## 6. Prediction scoreboard

Every prediction in `docs/predictions.md`, including any later amendments. **Statuses live in `docs/scoreboard.csv` and are changed by hand, never by code**; the default is "not yet run". A prediction that is contradicted, with the diagnostics that explain it, shows more understanding than a clean confirmation.

In [ ]:
report.scoreboard()

*Next: what the findings recommend, and how far they can be trusted.*

## 7. Verdict and limitations

**Use X when Y:** *to fill in after the full run.*

**Limitations.**

- **Low dimension:** the main setting is d = 2; the d = 5 check, if done, is one case.
- **Finite seeds:** 50 seeds gives a Wilson half-width of about 14% at a success rate near 0.5, so some differences will not be resolvable. A claim needs separated intervals; otherwise write "not distinguishable at this seed count".
- **One primary custom landscape**, plus two standard validation functions. A finding on the needle landscape need not transfer.
- **Regret needs a known minimum**, so success is defined only on benchmark functions, and $\varepsilon$ is a modelling choice (chosen from random sampling only).
- **Untested mechanisms are hypotheses.** Sigma collapse, lengthscale smoothing and selection averaging are explanations to be tested, not established facts.
- **Reproducibility:** A seed fixes the initial design and random streams. CMA-ES and random sampling reproduce exactly; BO reproduces in distribution, because floating-point differences compound through the model-fitting loop. Paired comparisons are paired on starting conditions only.
- **Frozen settings:** BO lengthscale bounds, CMA-ES start and restart triggers were fixed before the big runs; other choices could change the picture.